In [1]:
%load_ext autoreload
%autoreload 2

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["figure.facecolor"] = 'white'
matplotlib.rcParams["axes.facecolor"] = 'white'
matplotlib.rcParams["savefig.facecolor"] = 'white'
matplotlib.rcParams["text.color"] = 'black'
matplotlib.rcParams["axes.labelcolor"] = 'black'
matplotlib.rcParams["xtick.color"] = 'black'
matplotlib.rcParams["ytick.color"] = 'black'
matplotlib.rcParams["axes.edgecolor"] = 'black'
matplotlib.rcParams["axes.spines.right"] = False
matplotlib.rcParams["axes.spines.top"] = False
matplotlib.rcParams['font.family'] = 'cmss10'
matplotlib.rcParams['axes.formatter.use_mathtext'] = True
matplotlib.rcParams['mathtext.fontset'] = 'cm'

tick_fontsize = 18
label_fontsize = 20
title_fontsize = 28
legend_fontsize = 18
letter_fontsize = 21


cat_cmap = plt.get_cmap('Dark2')
cat_cmap_2 = plt.get_cmap('Pastel1')
cat_cmap_3 = plt.get_cmap('Accent')

sd_col = plt.get_cmap('Set2')(4)
hd_col = plt.get_cmap('Set2')(3)
ad_col = plt.get_cmap('Set2')(2)
av_col = plt.get_cmap('Set2')(1)
pos_col = plt.get_cmap('Set2')(0)

hd_cmap = plt.get_cmap('managua')
sd_cmap = plt.get_cmap('vanimo')

import o2s
import o2s.Tasks as Tasks

import gc
import sys
import numpy as np


/Users/jakelaherty/miniforge3/envs/O2S/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/jakelaherty/miniforge3/envs/O2S/lib/python3.12/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1711403213615/work/aten/src/ATen/native/TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [ ]:
task = o2s.task.Task.named('HD_SD-0D')
o2s.build.build(task, use_wandb=True)

wandb: Currently logged in as: jakelaherty (jakelaherty-gatsby-computational-neuroscience-unit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [7]:

from o2s.Tasks import SD_0D
checkpoint_path = '../trained-models/1741628198.4426634-task:SD-0D/checkpoint-epochs:2100/net.pt'

checkpoint = torch.load(checkpoint_path, map_location=device)
task = Task.from_checkpoint(checkpoint)
task.config.update(device=device, tau=1, dt=0.1)
torch.set_default_dtype(torch.float64 if task.config.precise else torch.float32)
net = RNN(task)
net.load_state_dict(checkpoint['net_state_dict'])
net.state_noise_std = 0

In [8]:
test = task.copy(av_step_zero_prob=0)
test_batch = TaskDataset(test).get_batch()
with torch.no_grad():
    sd_0d_test_sd_target = test_batch['vars']['sd'][0,:].cpu().numpy()
    sd_0d_test_states, _, sd_0d_test_outputs = net(test_batch['inputs'], noise=test_batch['noise'])
    sd_0d_test_states, sd_0d_test_outputs = sd_0d_test_states.cpu(), sd_0d_test_outputs.cpu()
    sd_0d_test_sd_pred = torch.atan2(sd_0d_test_outputs[0, :, 0], sd_0d_test_outputs[0, :, 1]).numpy()
    sd_0d_test_sd_pred[sd_0d_test_sd_pred < 0] += 2 * np.pi
del test, test_batch, sd_0d_test_outputs

T = np.linspace(0, task.config.n_timesteps/10, task.config.n_timesteps)
dt = T[1] - T[0]
sd_0d_test_sd_pred_x = []
sd_0d_test_sd_pred_y = []
for i, (t, sd) in enumerate(zip(T, sd_0d_test_sd_pred)):
    if i == 0:
        sd_0d_test_sd_pred_x.append(t)
        sd_0d_test_sd_pred_y.append(sd)
    else:
        diff = sd - sd_0d_test_sd_pred_y[-1]
        if np.abs(diff) > np.pi:
            sd_0d_test_sd_pred_x.extend([t - dt/2, t])
            sd_0d_test_sd_pred_y.extend([np.nan, sd])
        else:
            sd_0d_test_sd_pred_x.append(t)
            sd_0d_test_sd_pred_y.append(sd)

sd_0d_test_sd_target_x = []
sd_0d_test_sd_target_y = []
for i, (t, sd) in enumerate(zip(T, sd_0d_test_sd_target)):
    if i == 0:
        sd_0d_test_sd_target_x.append(t)
        sd_0d_test_sd_target_y.append(sd)
    else:
        diff = sd - sd_0d_test_sd_target_y[-1]
        if np.abs(diff) > np.pi:
            sd_0d_test_sd_target_x.extend([t - dt/2, t])
            sd_0d_test_sd_target_y.extend([np.nan, sd])
        else:
            sd_0d_test_sd_target_x.append(t)
            sd_0d_test_sd_target_y.append(sd)

In [ ]:
tuning_batch_size = 5000
tuning_vars_list = ['HD', 'ego_SD', 'allo_SD', 'AV']
n_diff_bins = 50
sd_0d_example_neurons = [0, 7]